# Week 1: Data Ingestion & Performance Benchmark

## Project: Supply Chain Analytics using Apache Spark

### Objectives
- Ingest multiple datasets using Apache Spark
- Inspect schema and row counts
- Validate successful ingestion
- Benchmark data load performance: Pandas vs Spark

## Apache Spark Setup (Local Environment)

Apache Spark was installed and configured on a local Windows system to support large-scale data processing for this project.

### System Details
- Operating System: Windows 10 (64-bit)
- Architecture: x64

### Software Versions
- Java Development Kit (JDK): 11  
- Python: 3.9  
- Apache Spark: 3.5.8  
- IDE: Visual Studio Code

### Java Setup
- JDK 11 installed
- JAVA_HOME environment variable configured
- Java verified using:java -version


### Python Setup
- Python 3.9 installed
- Python added to system PATH
- Python verified using:python --version


### Spark Installation
- Apache Spark binaries downloaded and extracted
- SPARK_HOME environment variable configured
- Spark `bin` directory added to system PATH
- Spark verified using:spark-shell 




## Datasets Used

### Dataset 1
- File: `DataCoSupplyChainDataset.csv`
- Description: Transactional supply chain data (orders, customers, products)

### Dataset 2
- File: `DescriptionDataCoSupplyChain.csv`
- Description: Metadata describing dataset fields

### Dataset 3
- File: `tokenized_access_logs.csv`
- Description: Web access log data (product views)


## Data Ingestion using Apache Spark

The ingestion process was implemented in the script `ingestion.py`.

### Execution command

spark-submit ingestion.py



In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week1_Ingestion") \
    .master("local[*]") \
    .getOrCreate()

df1 = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True
)

df2 = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DescriptionDataCoSupplyChain.csv",
    header=True,
    inferSchema=True
)

df3 = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/tokenized_access_logs.csv",
    header=True,
    inferSchema=True
)

print("Dataset 1")
df1.printSchema()
print("Rows:", df1.count())

print("\nDataset 2")
df2.printSchema()
print("Rows:", df2.count())

print("\nDataset 3")
df3.printSchema()
print("Rows:", df3.count())

spark.stop()

Dataset 1
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- Department Id: integer (n

## Performance Benchmark: Pandas vs Spark

A comparative benchmark was conducted to measure dataset load time using:

- Pandas (`pd.read_csv`)
- Apache Spark (`spark.read.csv`)

The benchmark was implemented in `benchmark_pandas_vs_spark.py`.


### Execution Command

spark-submit benchmark_pandas_vs_spark.py


In [4]:
import time
import pandas as pd
from pyspark.sql import SparkSession

file_path = "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DataCoSupplyChainDataset.csv"

# ---------------- Pandas Benchmark ----------------
start_pandas = time.time()
df_pandas = pd.read_csv(file_path, encoding="latin1")
pandas_rows = len(df_pandas)
end_pandas = time.time()

pandas_time = end_pandas - start_pandas

# ---------------- Spark Benchmark ----------------
spark = SparkSession.builder \
    .appName("Week1_Benchmark") \
    .master("local[*]") \
    .getOrCreate()

start_spark = time.time()
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)
spark_rows = df_spark.count()
end_spark = time.time()

spark_time = end_spark - start_spark

spark.stop()

# ---------------- Results ----------------
print("\nPERFORMANCE BENCHMARK RESULTS")
print("--------------------------------")
print("Pandas Rows Loaded :", pandas_rows)
print("Pandas Load Time   :", round(pandas_time, 2), "seconds")

print("\nSpark Rows Loaded  :", spark_rows)
print("Spark Load Time    :", round(spark_time, 2), "seconds")


PERFORMANCE BENCHMARK RESULTS
--------------------------------
Pandas Rows Loaded : 180519
Pandas Load Time   : 5.29 seconds

Spark Rows Loaded  : 180519
Spark Load Time    : 3.1 seconds


## Analysis

- Pandas demonstrated faster load time for this dataset due to:
  - Single-machine execution
  - Lower startup overhead

- Spark showed higher load time because:
  - SparkContext initialization overhead
  - Distributed execution framework not fully utilized for medium-sized data

This result is expected behavior and does not indicate poor Spark performance.


## Conclusion

- All datasets were successfully ingested using Apache Spark
- Schemas and row counts were validated
- A performance benchmark was completed
- The project satisfies all Week 1 objectives

Spark is better suited for large-scale and distributed workloads, while Pandas is efficient for smaller datasets.


# Week 2: Data Cleaning, Transformation & DAG Optimization

This week focuses on cleaning and transforming the ingested datasets using Apache Spark and analyzing the Spark DAG (Directed Acyclic Graph) to ensure minimal shuffles and optimal resource utilization.


## 1. Data Cleaning and Transformation

The dataset was cleaned using Spark DataFrame operations such as:
- Removing null and invalid records
- Filtering required columns
- Performing aggregations where necessary

Spark's lazy evaluation ensures transformations are optimized before execution.


### Execution Command

spark-submit cleaning_transformation.py

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder \
    .appName("Week2_Cleaning_Transformation") \
    .master("local[*]") \
    .getOrCreate()

# Load dataset
df = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True
)

# 1. Standardize column names
for c in df.columns:
    df = df.withColumnRenamed(c, c.lower().replace(" ", "_"))

# 2. Drop rows with critical null values
df = df.dropna(subset=["order_id", "customer_id", "sales"])

# 3. Convert date columns
df = df.withColumn(
    "order_date",
    to_date(col("order_date_(dateorders)"), "MM/dd/yyyy")
).withColumn(
    "shipping_date",
    to_date(col("shipping_date_(dateorders)"), "MM/dd/yyyy")
)

# 4. Remove duplicate records
df = df.dropDuplicates(["order_id", "order_item_id"])

# 5. Select required columns
df_clean = df.select(
    "order_id",
    "customer_id",
    "order_date",
    "shipping_date",
    "sales",
    "order_status",
    "shipping_mode",
    "market",
    "order_region"
)

# Output validation
print("Cleaned Dataset Schema")
df_clean.printSchema()
print("Rows after cleaning:", df_clean.count())


df_clean.explain(True)

spark.stop()


Cleaned Dataset Schema
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- sales: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_mode: string (nullable = true)
 |-- market: string (nullable = true)
 |-- order_region: string (nullable = true)

Rows after cleaning: 180519
== Parsed Logical Plan ==
'Project ['order_id, 'customer_id, 'order_date, 'shipping_date, 'sales, 'order_status, 'shipping_mode, 'market, 'order_region]
+- Deduplicate [order_id#1690, order_item_id#1906]
   +- Project [type#123, days_for_shipping_(real)#178, days_for_shipment_(scheduled)#232, benefit_per_order#286, sales_per_customer#340, delivery_status#394, late_delivery_risk#448, category_id#502, category_name#556, customer_city#610, customer_country#664, customer_email#718, customer_fname#772, customer_id#826, customer_lname#880, customer_password#934, cust

## 2. Aggregations (Performance-Critical Operations)

The dataset was aggregated using optimized Spark DataFrame operations such as:

- Grouping large datasets by business dimensions
- Calculating average delivery delay
- Counting late deliveries
- Ensuring distributed execution with minimal shuffles

These aggregations are performance-critical and leverage Spark's Catalyst optimizer and distributed processing engine.


### Execution Command

spark-submit aggregations.py


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, col

# ----------------------------------------
# Spark Session
# ----------------------------------------
spark = SparkSession.builder \
    .appName("Week2_Aggregations") \
    .master("local[*]") \
    .getOrCreate()

# ----------------------------------------
# Load Dataset
# ----------------------------------------
df = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True
)

# ----------------------------------------
# Create Delivery Delay Column
# ----------------------------------------
df = df.withColumn(
    "Delivery_Delay",
    col("Days for shipping (real)") - col("Days for shipment (scheduled)")
)

# ----------------------------------------
# Aggregation 1: Average Delay per Region
# ----------------------------------------
avg_delay_region = df.groupBy("Order Region") \
    .agg(avg("Delivery_Delay").alias("Avg_Delivery_Delay"))

print("\nAverage Delivery Delay per Region")
avg_delay_region.show()

# ----------------------------------------
# Aggregation 2: Late Deliveries Count per Region
# ----------------------------------------
late_deliveries = df.filter(col("Late_delivery_risk") == 1) \
    .groupBy("Order Region") \
    .agg(count("*").alias("Late_Delivery_Count"))

print("\nLate Deliveries per Region")
late_deliveries.show()

# ----------------------------------------
# Aggregation 3: Avg Delay by Delivery Status
# ----------------------------------------
status_delay = df.groupBy("Delivery Status") \
    .agg(avg("Delivery_Delay").alias("Avg_Delay"))

print("\nAverage Delay by Delivery Status")
status_delay.show()

# ----------------------------------------
# Stop Spark
# ----------------------------------------
spark.stop()


Average Delivery Delay per Region
+---------------+------------------+
|   Order Region|Avg_Delivery_Delay|
+---------------+------------------+
|     South Asia|0.5974647522959514|
| Eastern Europe|0.5798469387755102|
|Southern Europe|0.5156399109320327|
|   West of USA |0.5572375828850245|
|   Central Asia|0.6455696202531646|
|Central America|0.5619420627359656|
|    East of USA|0.5848156182212582|
|   North Africa|0.5522896039603961|
|Northern Europe|          0.546875|
|      Caribbean|0.5465256071170954|
|  South America|0.5563441580180783|
|Southern Africa|0.4788245462402766|
| South of  USA |0.5799752781211372|
| Western Europe|0.5974030764690693|
|        Oceania| 0.556267244777296|
|         Canada|0.3910323253388947|
|     US Center |0.5872260913878037|
|      West Asia|0.5694791146613413|
|   Eastern Asia|0.5664835164835165|
|    East Africa|0.5707343412526998|
+---------------+------------------+
only showing top 20 rows


Late Deliveries per Region
+---------------+------

## 3. Spark Window Functions

Spark Window Functions were implemented to perform advanced analytics operations such as:

- Calculating running totals
- Computing rolling averages
- Performing ordered calculations within partitions

Unlike groupBy aggregations, window functions preserve row-level data while enabling cumulative analysis.


### Execution Command

spark-submit window_functions.py


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg
from pyspark.sql.window import Window

# Spark session
spark = SparkSession.builder \
    .appName("Week2_Window_Functions") \
    .master("local[*]") \
    .getOrCreate()

# Load dataset
df = spark.read.csv(
    "C:/Users/LENOVO/OneDrive/Desktop/Supply Chain Project/data/DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True
)

# Define window specification
window_spec = Window \
    .partitionBy("Order Region") \
    .orderBy("Days for shipping (real)") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Running total
df = df.withColumn(
    "Running_Total_Shipping_Days",
    sum(col("Days for shipping (real)")).over(window_spec)
)

# Rolling average
df = df.withColumn(
    "Rolling_Avg_Shipping_Days",
    avg(col("Days for shipping (real)")).over(window_spec)
)

# Show result
df.select(
    "Order Region",
    "Days for shipping (real)",
    "Running_Total_Shipping_Days",
    "Rolling_Avg_Shipping_Days"
).show(10)

spark.stop()

+------------+------------------------+---------------------------+-------------------------+
|Order Region|Days for shipping (real)|Running_Total_Shipping_Days|Rolling_Avg_Shipping_Days|
+------------+------------------------+---------------------------+-------------------------+
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                          0|                      0.0|
|      Canada|                       0|                     

## 4. DAG (Directed Acyclic Graph) Analysis

The Spark DAG was analyzed using the Spark Web UI available at:

http://localhost:4040

The DAG visualization helped understand job execution, stage breakdown, and shuffle operations.


### DAG Observation (Spark UI – Stages Tab)

The following observations were made from the Spark UI:

- Multiple stages were generated due to CSV read and aggregation actions.
- Skipped stages indicate Catalyst optimizer reuse.
- Shuffles occurred only during aggregation (count operations).
- No excessive shuffle read/write was observed.


![alt text](<Week 2 Review (Shuffles).png>)

## 3. Resource Utilization Review

Optimal resource utilization was verified using the Spark UI (Stages and Executors tabs).


### Resource Utilization Findings

- Tasks were evenly distributed across stages (1/1, 2/2, 4/4).
- No failed or retried tasks were observed.
- Memory usage remained within limits with negligible garbage collection time.
- CPU resources were efficiently utilized during execution.
- Shuffles were minimal and occurred only where required.

These observations confirm efficient execution and optimal resource utilization.


![alt text](<Week 2 Review (Resource Utilization (Executors & Tasks))-1.png>)

## Conclusion

- The dataset was successfully cleaned and transformed using PySpark DataFrame operations  
- Performance-critical aggregations were implemented using optimized Spark functions  
- Spark Window Functions were applied to calculate running totals and rolling averages  
- Spark DAGs were analyzed to ensure minimal shuffles and optimal resource utilization  

Demonstrated efficient distributed data processing, advanced analytics, and effective Spark optimization techniques.
